In [43]:
import pandas as pd
import glob
from datetime import date, timedelta
import numpy as np
from datetime import datetime
import pathlib
from pathlib import Path
from collections import OrderedDict
import polars as pl
import fastexcel
import os
import time

In [44]:
def convert_to_datetime(struct_time):
    """Convert struct_time to datetime object."""
    return datetime(*struct_time[:6])

def input_data(folder_path, sheet_name=None):
    file_paths = glob.glob(f"{folder_path}/*.xlsx") + glob.glob(f"{folder_path}/*.csv")
    df_list = []

    for file in file_paths:
        export_time = os.path.getmtime(file)
        export_time_datetime = convert_to_datetime(time.localtime(export_time))

        if file.endswith('.xlsx'):
            df = pl.read_excel(file, sheet_name=sheet_name)
        elif file.endswith('.csv'):
            try:
                df = pl.read_csv(file, encoding="utf-8", infer_schema_length=5000)
            except:
                df = pl.read_csv(file, encoding="ISO-8859-1", ignore_errors=True, infer_schema_length=5000)

        df = df.with_columns([
            pl.col(col).cast(pl.String)
            for col in df.columns
        ])

        df = df.with_columns([
            pl.lit(os.path.basename(file)).alias('File Name'),
            pl.lit(export_time_datetime).alias('Export Time')
        ])

        df_list.append(df)

    if df_list:
        merged_df = pl.concat(df_list, how='vertical')
    else:
        merged_df = pl.DataFrame()

    return merged_df

def input_data_parquet(folder_path):
    file_paths = glob.glob(f"{folder_path}/*.parquet")
    df_list = []

    for file in file_paths:
        export_time = os.path.getmtime(file)
        export_time_datetime = convert_to_datetime(time.localtime(export_time))

        df = pl.read_parquet(file)

        df = df.with_columns([
            pl.lit(os.path.basename(file)).alias('File Name'),
            pl.lit(export_time_datetime).alias('Export Time')
        ])
        df_list.append(df)

    if df_list:
        merged_df = pl.concat(df_list, how='vertical')
    else:
        merged_df = pl.DataFrame()

    return merged_df
        
today_temp = datetime.today().date()
today = today_temp.strftime('%b_%d_%Y')

def pre_process_fcr_excel(folder_path: str):
    excel_files = glob.glob(os.path.join(folder_path, "*.xlsx")) + glob.glob(os.path.join(folder_path, "*.xls"))
    
    for file_path in excel_files:
        try:
            df = pl.read_excel(file_path, engine="calamine", has_header=False)
            
            if df.height < 2:
                continue
            header_row_index = 0
            for i in range(min(10, df.height)):
                row_data = [str(x).strip() for x in df.row(i) if x is not None]
                if "FCR Category" in row_data or "Conversation Id" in row_data:
                    header_row_index = i
                    break
            raw_headers = df.row(header_row_index)
            headers = []
            seen = set()
            for i, c in enumerate(raw_headers):
                col_name = str(c).strip() if c is not None and str(c).strip() != "" else f"col_{i}"
                if col_name in seen:
                    col_name = f"{col_name}_dup_{i}"
                seen.add(col_name)
                headers.append(col_name)

            df = df.slice(header_row_index + 1)
            df.columns = headers
            csv_path = os.path.splitext(file_path)[0] + ".csv"
            df.write_csv(csv_path)
            # os.remove(file_path)
        except Exception:
            pass

In [45]:
first_glob = os.path.expanduser("~").replace("\\", "/")
test_path = f"{first_glob}/Concentrix Corporation"
if not os.path.exists(test_path):
    raise FileNotFoundError(f"Not found the path: {test_path}")

folder_paths = {
    "input_performance":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/NEW_LOOK_EXCEL_EN/new_look_excel_data',
    "output_miv_performance":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/MIV/MIV_Data',
    "hc_extend_by_month":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Extend by Month',
    "input_survey":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/SURVEY_EN',
    "input_afcr":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/FCR',
    "input_t3":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/T3_EN',
    "input_iex":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_OUTPUT_AGENT_IEX',
    "mapping_file": f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/AQG_summarized.xlsx',
    "global_hc": f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources/Global_HC.parquet',
    "input_parquet_performance":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/NEW_LOOK_EXCEL_EN/parquet_rawdata',
    "input_csv_pulse":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/Detractor_Compliance/0_source',
    "output_csv_pulse":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/Detractor_Compliance/1_des',
    "input_xlsx_qa_audit":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/Detractor_Compliance/2_qa_audit',
    "input_csv_re_direct":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/RE-DIRECT',
    "input_delayed_closure":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/DELAYED_CLOSURE',
    "output_csv_verbatim":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/VERBATIM',
}

print("--- FULL FOLDER PATHS LIST ---")
for key, path in folder_paths.items():
    print(f"{key}: {path}")

print("-" * 60)

pre_process_fcr_excel(folder_paths["input_afcr"])

IEX = input_data(folder_paths["input_iex"])
IEX = IEX.unique()
IEX = IEX.with_columns([
    pl.col(['Date']).str.to_date("%Y-%m-%d", strict=False),
    pl.col(['Datetime_Fluctuate_Start_Shift','Datetime_Fluctuate_End_Shift','Datetime_First_Start_Shift','Datetime_First_End_Shift']).str.to_datetime("%Y-%m-%d %H:%M:%S.%f", strict=False),
    pl.col(['Night_Shift','Target','Unplanned','Planned','Roster Presented','Roster Scheduled']).cast(pl.Float64)])
columns_to_sec = ['Time_Of_Day','Open Time', 'Extra Time', 'Break Time', 'Lunch Time', 'Training', 'NCNS','AL','Target']
IEX = IEX.with_columns([(pl.col(col).fill_null(0).cast(pl.Float64) * 3600).alias(col) for col in columns_to_sec])
Night_Shift_1 = IEX[['Date','Email Id','Night_Shift']].unique()
Night_Shift_2 = Night_Shift_1.with_columns((pl.col('Date') - pl.duration(days=1)).alias('Previous Date'))
Night_Shift = Night_Shift_2.join(Night_Shift_1,left_on = ['Previous Date','Email Id'], right_on = ['Date','Email Id'], how = 'left')
Night_Shift = Night_Shift.rename({'Night_Shift_right':'Previous_Night_Shift'})
Previous_Date = IEX[['Date','Email Id','Datetime_First_Start_Shift','Shift Tracking']].unique()

--- FULL FOLDER PATHS LIST ---
input_performance: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/NEW_LOOK_EXCEL_EN/new_look_excel_data
output_miv_performance: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/MIV/MIV_Data
hc_extend_by_month: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Extend by Month
input_survey: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/SURVEY_EN
input_afcr: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/FCR
input_t3: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/T3_EN
input_iex: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_OUTPUT_AGENT_IEX
mapping_file: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/

Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43,

In [46]:
SURVEY_INPUT = input_data(folder_paths["input_survey"])

survey_added_ir = SURVEY_INPUT.with_columns([
    pl.when(
        (pl.col("Question Category") == "Resolution") & (pl.col("Answer") == "Yes")
    ).then(1)
    .when(
        (pl.col("Question Category") == "Resolution") & (pl.col("Answer") == "No")
    ).then(0)
    .otherwise(0)
    .alias("_ir")
])

survey_ae = (
    SURVEY_INPUT
    .filter(pl.col("Question Category") == "agentExperience")
    .with_columns(
        pl.col("Answer")
          .cast(pl.Int64, strict=False)
          .alias("_ae")
    )
)

survey_verbatim = (
    SURVEY_INPUT
    .filter(pl.col("Question Category") == "Feedback")
    .with_columns([
        pl.col("Answer").str.replace_all(r"[\r\n\t]+", " ").alias("_verbatim"),
        pl.concat_str([
            pl.col("Agent Email ID").cast(pl.String).fill_null(""),
            pl.col("Conversation Id").cast(pl.String).fill_null(""),
            pl.col("Joined Time").str.to_datetime(strict=False).dt.strftime("%y%m%d%H%M%S")
        ], separator="_").alias("_conver_unique"),
    ])
)

metrics = ["d_happy_response", "d_surprised_response", "e_response", "t_response", "u_response"]

survey_duet = (
    SURVEY_INPUT
    .filter(pl.col("Question Category").is_in(metrics))
    .with_columns([
        pl.col("Question Category").alias("metric"),
        pl.col("Answer").cast(pl.Float64, strict=False).alias("value")
    ])
    .group_by(["Conversation Id", "metric"])
    .agg(pl.max("value").alias("value")) 
    .pivot(values="value", index="Conversation Id", columns="metric")
)

# --- Excel-equivalent calculations using column names (force Float64) ---
survey_duet = survey_duet.with_columns([
    pl.when(pl.any_horizontal([pl.col("d_happy_response").is_null(),
                               pl.col("d_surprised_response").is_null()]))
      .then(pl.lit(None, dtype=pl.Float64))
      .otherwise(pl.when((pl.col("d_happy_response") >= 4) & (pl.col("d_surprised_response") >= 3))
                   .then(100.0).otherwise(0.0))
      .alias("delight"),

    pl.when(pl.col("u_response").is_null()).then(pl.lit(None, dtype=pl.Float64))
      .otherwise(((pl.col("u_response") - 1.0) / 6.0) * 100.0).alias("usability"),

    pl.when(pl.col("e_response").is_null()).then(pl.lit(None, dtype=pl.Float64))
      .otherwise(((pl.col("e_response") - 1.0) / 6.0) * 100.0).alias("ease"),

    pl.when(pl.col("t_response").is_null()).then(pl.lit(None, dtype=pl.Float64))
      .otherwise(((pl.col("t_response") - 1.0) / 6.0) * 100.0).alias("trust"),

    pl.when(pl.any_horizontal([pl.col("d_happy_response").is_null(),
                               pl.col("d_surprised_response").is_null(),
                               pl.col("u_response").is_null(),
                               pl.col("e_response").is_null(),
                               pl.col("t_response").is_null()]))
      .then(pl.lit(None, dtype=pl.Float64))
      .otherwise(pl.when((pl.col("d_happy_response") >= 4) &
                         (pl.col("d_surprised_response") >= 3) &
                         ((pl.col("u_response") + pl.col("e_response")) >= 13) &
                         (pl.col("e_response") >= 6) &
                         (pl.col("t_response") >= 6))
                   .then(100.0).otherwise(0.0))
      .alias("DUET"),
])

# Round computed columns
survey_duet = survey_duet.with_columns([
    pl.col("delight").round(0),
    pl.col("usability").round(0),
    pl.col("ease").round(0),
    pl.col("trust").round(0),
    pl.col("DUET").round(0),
])

# Keep ALL required columns (raw + computed)
final_cols = ["Conversation Id"] + metrics + ["delight", "usability", "ease", "trust", "DUET"]
survey_duet = survey_duet.select([c for c in final_cols if c in survey_duet.columns])
survey_ir = (survey_added_ir.select(["Conversation Id", "_ir"]).filter(pl.col("_ir") == 1).unique())
survey_ae = (survey_ae.select(["Conversation Id", "_ae"]).filter(pl.col("_ae") > 0).unique())
verbatim = (survey_verbatim.select(["_conver_unique", "_verbatim"])).unique()

C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_26692\1549389880.py:48: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  .pivot(values="value", index="Conversation Id", columns="metric")


In [47]:
T3_INPUT = input_data(folder_paths["input_t3"])

t3_added_col_t3 = T3_INPUT.with_columns([
    pl.when(
        (pl.col("Agent Queue Group Name").str.contains("T3", literal=True)) &
        (pl.col("Transferred (Yes/No)") == "Yes")
    ).then(1).otherwise(0).alias("T3")])

t3_final = (t3_added_col_t3.select(["Conversation Id", "T3"]).filter(pl.col("T3") == 1).unique())
t3_final

Conversation Id,T3
str,i32
"""7ba90844-5f45-46b6-b953-cf70e4…",1
"""c095daf2-c7a3-44aa-ba1a-398179…",1
"""44cc6414-d039-43c1-a00a-5ebdf3…",1
"""5ee06cb6-4bbc-4db2-938a-6bd738…",1
"""e372fbf8-5ca6-491e-a566-f199e1…",1
…,…
"""84ef8109-a73e-4107-8897-b82972…",1
"""7e77dd82-1668-4d27-a7fb-61adb2…",1
"""fba953fa-b609-47a0-a71c-63cf3d…",1


In [48]:
def process_afcr_folder(folder_path: str) -> pl.DataFrame:
    all_dataframes = []
    
    csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
    
    for file_path in csv_files:
        file_name = os.path.basename(file_path)
        timestamp = os.path.getmtime(file_path)
        export_time = datetime.fromtimestamp(timestamp).strftime('%Y-%m-%d %H:%M:%S')
        
        df = pl.read_csv(
            file_path, 
            encoding="utf-8", 
            schema_overrides={"Itinerary": pl.String},
            infer_schema_length=10000,
            ignore_errors=True
        )
        
        cast_exprs = []
        for col, dtype in [("Handle Time", pl.Float64), ("Duet", pl.Float64), 
                           ("Passed Sessions", pl.Int64), ("Failed Sessions", pl.Int64)]:
            if col in df.columns:
                cast_exprs.append(pl.col(col).cast(dtype, strict=False))
                
        if cast_exprs:
            df = df.with_columns(cast_exprs)
        
        df = df.with_columns([
            pl.lit(file_name).alias('File Name'),
            pl.lit(export_time).alias('Export Time')
        ])
        all_dataframes.append(df)
        
    if not all_dataframes:
        return pl.DataFrame()
        
    return pl.concat(all_dataframes, how="diagonal_relaxed")


afcr_input = process_afcr_folder(folder_paths["input_afcr"])

if not afcr_input.is_empty():
    afcr_input = (
        afcr_input
        .filter(pl.col("Vendor Partner Location") == "Concentrix (Ho Chi Minh City)")
        .select([
            pl.col("Agent Email Address").alias("Agent Email ID"),
            pl.col("Conversation Id"),
            pl.col("Passed Sessions"),
            pl.col("Failed Sessions"),
            pl.when(pl.col("Passed Sessions").is_in([0, 1]))
            .then(1)
            .otherwise(0)
            .alias("Total Sessions")
        ])
        .unique()
    )

In [49]:
DELAYED_CLOSURE_INPUT = (
    input_data(folder_paths["input_delayed_closure"])
    .filter(pl.col("Agent Vendor Location") == "Concentrix (Ho Chi Minh City)")
    .unique(subset=["Agent Email ID", "Conversation Id", "Joined Time"], keep='last'))
delayed_closure = (
    DELAYED_CLOSURE_INPUT
    .select([
        "Agent Email ID", "Conversation Id", "Joined Time", "Traveler Unresponsive Time", "Response Time",
        "Agent Disconnect (Yes / No)", "Ghost (Yes / No)", "Requeued (Yes / No)"
    ])
    .with_columns([
        pl.col("Joined Time").str.to_datetime().cast(pl.Datetime),
        pl.col("Traveler Unresponsive Time").cast(pl.Float64),
        pl.col("Response Time").cast(pl.Float64),
        (pl.col("Traveler Unresponsive Time").cast(pl.Float64) - 240)
            .clip(0, None)
            .alias("Exceed Time"),
        ((pl.col("Traveler Unresponsive Time").cast(pl.Float64) - 240) > 0)
            .cast(pl.Int8)
            .alias("Exceed Chat"),
        (pl.col("Agent Disconnect (Yes / No)") == "Yes").cast(pl.Int8).alias("Agent Disconnect"),
        (pl.col("Ghost (Yes / No)") == "Yes").cast(pl.Int8).alias("Ghost"),
        (pl.col("Requeued (Yes / No)") == "Yes").cast(pl.Int8).alias("Requeued"),
        (pl.col("Response Time").cast(pl.Float64).is_null() | (pl.col("Response Time").cast(pl.Float64) == 0))
            .cast(pl.Int8)
            .alias("Traveler Unresponsive")
    ])
    .drop([
        "Agent Disconnect (Yes / No)", "Ghost (Yes / No)", "Requeued (Yes / No)",
        "Traveler Unresponsive Time", "Response Time"
    ])
    .group_by(["Agent Email ID", "Conversation Id", "Joined Time"])
    .agg([
        pl.sum("Exceed Time").alias("Exceed Time"),
        pl.sum("Exceed Chat").alias("Exceed Chat"),
        pl.sum("Agent Disconnect").alias("Agent Disconnect"),
        pl.sum("Ghost").alias("Ghost"),
        pl.sum("Requeued").alias("Requeued"),
        pl.sum("Traveler Unresponsive").alias("Traveler Unresponsive")
    ])
)

delayed_closure = delayed_closure.with_columns(
    pl.when(pl.col("Exceed Time") <= 0).then(pl.lit(None, dtype=pl.Utf8))
      .when((pl.col("Exceed Time")/60) <= 1).then(pl.lit("01 Mins"))
      .when((pl.col("Exceed Time")/60) <= 2).then(pl.lit("02 Mins"))
      .when((pl.col("Exceed Time")/60) <= 3).then(pl.lit("03 Mins"))
      .when((pl.col("Exceed Time")/60) <= 4).then(pl.lit("04 Mins"))
      .when((pl.col("Exceed Time")/60) <= 5).then(pl.lit("05 Mins"))
      .when((pl.col("Exceed Time")/60) <= 10).then(pl.lit("05-10 Mins"))
      .when((pl.col("Exceed Time")/60) <= 15).then(pl.lit("10-15 Mins"))
      .when((pl.col("Exceed Time")/60) <= 30).then(pl.lit("15-30 Mins"))
      .otherwise(pl.lit("30+ Min"))
      .alias("Exceed Bucket")
).unique()

print(delayed_closure.height)

49643


In [50]:
# def input_csv_pulse(folder_path: str) -> pl.DataFrame:
#     selected_columns = [
#         "AUDIT ID", "AUDITEE NAME", "AUDITEE EMPLOYEE ID", "AUDITEE SUPERVISOR NAME",
#         "AUDITEE SUPERVISOR EMPLOYEE ID", "AUDITOR NAME", "AUDITOR EMPLOYEE ID",
#         "DELEGEE NAME", "DELEGEE EMPLOYEE ID", "AUDIT DATE", "LOBS", "TRANSACTION ID",
#         "TRANSACTION DATE", "MONITORING TYPE", "ITINERARY NUMBER", "RESOLUTION AVAILABILITY",
#         "TRAVELER'S PAIN POINT", "ARTICLE NUMBER THAT SHOULD HAVE BEEN FOLLOWED?", "LOB #",
#         "PASTE THE SUB-TOPIC FROM ARTICLE (E.G. PROPERTY SAYS NO (INSIDE PENALTY))",
#         "LANGUAGE..", "TRIP STAGE.", "SUB - INTENT", "PRODUCT CATEGORY",
#         "GENERAL FEEDBACK ON THE AUDIT", "ACKNOWLEDGEMENT/DISPUTE DATE", "AUDIT STATUS",
#         "GENERAL COMMENT ON DISPUTE", "IS ESCALATED", "IS RESOLVED", "COACHING SCHEDULED",
#         "COACHING PUBLISHED DATE", "COACHING ACKNOWLEDGEMENT STATUS",
#         "AUDIT LAST MODIFIED DATE", "AUDIT MODIFIED BY", "ACTUAL AUDITOR"
#     ]

#     dfs = []

#     for filename in os.listdir(folder_path):
#         if filename.endswith(".csv"):
#             file_path = os.path.join(folder_path, filename)
#             print(f"📄 Đang đọc file: {file_path}")
#             df = pd.read_csv(file_path)
#             existing_cols = [col for col in selected_columns if col in df.columns]
#             df = df[existing_cols]
#             if "TRANSACTION ID" in df.columns:
#                 df["TRANSACTION ID"] = df["TRANSACTION ID"].astype(str).str.replace("'", "", regex=False)
#             if "AUDITEE EMPLOYEE ID" in df.columns and "TRANSACTION ID" in df.columns:
#                 df["OracleID_ConversationID_KEY"] = df["AUDITEE EMPLOYEE ID"].astype(str) + "_" + df["TRANSACTION ID"].astype(str)
#             dfs.append(df)

#     merged_df = pd.concat(dfs, ignore_index=True)
#     pl_df = pl.from_pandas(merged_df)

#     return pl_df

# PULSE_INPUT = input_csv_pulse(folder_paths["input_csv_pulse"])
# PULSE_INPUT

In [51]:
# QA_AUDIT_INPUT = input_data(folder_paths["input_xlsx_qa_audit"])

# QA_AUDIT_INPUT = QA_AUDIT_INPUT.with_columns([
#     pl.concat_str([
#         pl.col("Agent Email").cast(str),
#         pl.col("Conversation ID").cast(str),
#     ], separator="_")
#     .alias("EmailID_ConversationID_KEY"),
#     pl.lit(1).alias("QA_Audit")
# ])
# QA_AUDIT_INPUT_COLS = ["EmailID_ConversationID_KEY", "Case #", "AutoFail", "Evaluator Name", "Evaluation Date" ,"QA_Audit"]
# QA_AUDIT_INPUT = QA_AUDIT_INPUT.select(QA_AUDIT_INPUT_COLS)
# QA_AUDIT_INPUT

In [52]:
RE_DIRECT_INPUT = (
    input_data(folder_paths["input_csv_re_direct"])
    .with_columns(
        PeopleID_ConversationID_KEY=pl.concat_str(
            [pl.col("Agent People Id").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],
            separator="_",
        ),
        Re_Direct=pl.lit(1),
        **{
            "Re-Direct Text": (
                pl.col("Text").cast(pl.Utf8).fill_null("")
                .str.replace_all(r"(\r\n|\r|\n)+", " | ")
                .str.replace_all(r"[\-•\u2022\u25CF\u25E6\u2043\u2219\u00B7\u2013\u2014]+", "")
                .str.replace_all(r"\s{2,}", " ")
                .str.strip_chars()
            )
        }
    )
    .select(["PeopleID_ConversationID_KEY", "Re_Direct", "Re-Direct Text"])
    .unique(maintain_order=True)
)

In [ ]:
PERFORMANCE_INPUT = input_data(folder_paths["input_performance"])



try: 
    if PERFORMANCE_INPUT.columns[0] == "":
        PERFORMANCE_INPUT = PERFORMANCE_INPUT.drop(PERFORMANCE_INPUT.columns[0])
except: pass

print(PERFORMANCE_INPUT.columns)
existing_cols = set(PERFORMANCE_INPUT.columns)

columns_to_cast = {
    "Joined Time": pl.Datetime,
    "Handle Time (Sum)": pl.Float64,
    "Handle (Count)": pl.Int64,
    "Hold Time (Sum)": pl.Float64,
    "Talk Time (Sum)": pl.Float64,
    "Wrap Up Time (Sum)": pl.Float64,
    "Survey Submitted (Count)":pl.Float64,
    "Promoter Score (Calc)": pl.Float64,
    "Detractor Score (Calc)": pl.Float64,
    "Neutral Score (Calc)": pl.Float64,
    "Has Followup Time": pl.Float64,
    "Response Time (Sum)": pl.Float64,
    "Response Time (Avg)": pl.Float64,
    "Concurrency": pl.Float64,
    "Survey Offered (Count)":pl.Int64,
}

datetime_cols = ["Started Time", "Joined Time", "Submitted Time", "Left Time"]

casts = []

for col, dtype in columns_to_cast.items():
    if col in existing_cols:
        if col == "Submitted Date":
            casts.append(pl.col(col).str.strptime(pl.Date, "%m/%d/%Y", strict=False))
        elif col in datetime_cols:
            casts.append(
                pl.when(pl.col(col).str.contains(r"^\d{4}-\d{2}-\d{2}"))  # ISO: YYYY-MM-DD
                  .then(pl.col(col).str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False))
                .when(pl.col(col).str.contains(r"^\d{1,2}/\d{1,2}/\d{4}"))  # US: M/D/YYYY
                  .then(pl.col(col).str.strptime(pl.Datetime, "%m/%d/%Y %H:%M", strict=False))
                .otherwise(None)
                .alias(col)
            )
        elif dtype == pl.Float64:
            casts.append(
                pl.col(col)
                .cast(pl.Utf8)
                .str.replace_all(",", "")
                .cast(pl.Float64, strict=False)
                .alias(col)
            )
        else:
            casts.append(pl.col(col).cast(dtype).alias(col))

PERFORMANCE_CHANGED_TYPE = PERFORMANCE_INPUT.with_columns(casts)
PERFORMANCE_ADDED_CCR72 = PERFORMANCE_CHANGED_TYPE.with_columns([
    pl.when(pl.col("Has Followup Time").cast(pl.Float64) <= 4320)
      .then(1.0)
      .otherwise(0.0)
      .alias("CCR72")
])

PERFORMANCE_NEXT_STEP = PERFORMANCE_ADDED_CCR72.with_columns([
    pl.col("Joined Time").dt.date().alias("Joined Date")
])

PERFORMANCE_NEXT_STEP = PERFORMANCE_NEXT_STEP.with_columns([
    (pl.col("Joined Time") + pl.duration(hours=14)).alias("Join Time (VNT)"),
    (pl.col("Joined Time") + pl.duration(hours=14)).dt.date().alias("Join Date (VNT)")
])

HC_MASTER_DATABASE = input_data(folder_paths["hc_extend_by_month"])
HC_MASTER_DATABASE = HC_MASTER_DATABASE.rename({'Date Start Week': 'Week_Monday'})

HC_MASTER_DATABASE = HC_MASTER_DATABASE.with_columns([
    pl.col('Date').str.strptime(pl.Date, "%Y-%m-%d", strict=False)
])

hc_master_selected = HC_MASTER_DATABASE.select([
    "Date","Email Id", "OracleID", "People ID", "IEX ID", "Employee Name", "Alias", "Designation", "Detail Status", "Active", "TL ID", "Supervisor Email",
    "Supervisor Name", "Wave", "LOB", 'LG Tenure', 'NL Tenure', 'Mini TL - Email', 'Mini TL - Short Name', 'Mini TL Start Date', 'Site'
]).unique()

hc_master_selected = hc_master_selected.rename({'Mini TL - Short Name': 'Mini TL', 'LOB':'Group'})
performance_merged = PERFORMANCE_NEXT_STEP.join(
    hc_master_selected,
    left_on=["Joined Date","Agent Email ID"],
    right_on=["Date","Email Id"],
    how="left")
GLOBAL_HC = pl.read_parquet(folder_paths["global_hc"])
global_hc_clean = GLOBAL_HC.select(["SSO ID","Production Start date","Agent/Non Agent"]).unique(subset=["SSO ID"], keep="first")
merged_global_hc = performance_merged.join(global_hc_clean,left_on="Agent Email ID",right_on="SSO ID",how="left")
merged_iex = merged_global_hc.join(IEX[['Date','Email Id','First Shift','Datetime_First_Start_Shift','Night_Shift']], left_on=['Join Date (VNT)','Agent Email ID'], right_on=['Date','Email Id'], how='left')
mapping = pl.read_excel(folder_paths["mapping_file"])
performance_cleaned = merged_iex.join(mapping, on="Agent Queue Group Name",how='left')
lc_mapping = pl.read_excel(folder_paths["mapping_file"], sheet_name="kpi")

['Joined Date', 'Agent People Id', 'Conversation Id', 'Agent Email ID', 'Agent Queue Group Name', 'Latest VA Intent', 'Agent Business Location', 'Initiated Outbound (Yes / No)', 'Latest VA Product', 'Response Count', 'Response Time', 'Customer Loyalty Status', 'Business Segment Name', 'Partner Name', 'Language', 'Effective Disconnected By', 'Joined Time', 'Left Time', 'Entry Point App', 'Device Category', 'Device Os', 'Browser', 'Has Followup Time', 'Queue Name', 'Agent Manager Name', 'Agent Name', 'Handle (Count)', 'Handle Time (Sum)', 'Wrap Up Time (Sum)', 'Hold Time (Sum)', 'Talk Time (Sum)', 'Follow Up (Count)', 'Survey Submitted (Count)', 'Promoter Score (Calc)', 'Detractor Score (Calc)', 'Assigned Agent Time (Sum)', 'Sum of Chat Agent First Response Time', 'Survey Offered (Count)', 'File Name', 'Export Time']


Could not determine dtype for column 5, falling back to string


In [54]:
performance_updated_ns = performance_cleaned.with_columns((pl.col('Join Date (VNT)') - pl.duration(days=1)).alias('Previous Date'))
performance_updated_ns = performance_updated_ns.join(Night_Shift[['Date','Email Id','Night_Shift','Previous_Night_Shift']],
                                                                 left_on=['Join Date (VNT)','Agent Email ID'],
                                                                 right_on=['Date','Email Id'],
                                                                 how='left')
def update_night_shift(df: pl.DataFrame) -> pl.DataFrame:
    df = df.with_columns(
        pl.when(
            (pl.col('Night_Shift') == 0) & 
            (pl.col('Join Time (VNT)').dt.time() < pl.time(17, 0)) & 
            (pl.col('Join Time (VNT)').dt.time() >= pl.time(0, 0)) & 
            (pl.col('Previous_Night_Shift') == 1)
        ).then(False).otherwise(True).alias('Night_Shift_2_Check')
    )

    df = df.with_columns(
        pl.when(pl.col('Night_Shift_2_Check') == False)
        .then(1)
        .otherwise(pl.col('Night_Shift'))
        .alias('Night_Shift')
    )

    df = df.with_columns(
        pl.when((pl.col('Join Time (VNT)').dt.hour() >= 0) & (pl.col('Join Time (VNT)').dt.hour() < 12) & (pl.col('Previous_Night_Shift') == 1))
        .then(pl.col('Join Date (VNT)') - pl.duration(days=1))
        
        .when((pl.col('Join Time (VNT)').dt.hour() >= 0) & (pl.col('Join Time (VNT)').dt.hour() < 12) & (pl.col('Night_Shift') == 1))
        .then(pl.col('Join Date (VNT)') - pl.duration(days=1))
        
        .when((pl.col('Join Time (VNT)').dt.hour() >= 0) & (pl.col('Join Time (VNT)').dt.hour() < 18) & (pl.col('Night_Shift') == 0))
        .then(pl.col('Join Date (VNT)'))
        
        .when((pl.col('Join Time (VNT)').dt.hour() >= 18) & (pl.col('Night_Shift') == 1))
        .then(pl.col('Join Date (VNT)'))
        
        .otherwise(pl.col('Join Date (VNT)'))
        .alias('_Date_Converted')
    )

    return df

performance_updated_ns = update_night_shift(performance_updated_ns)
performance_updated_ns.select(([
    "Joined Time",
    "Join Time (VNT)",  
    "Join Date (VNT)",
    "_Date_Converted"
]))

Joined Time,Join Time (VNT),Join Date (VNT),_Date_Converted
datetime[μs],datetime[μs],date,date
2026-04-30 12:40:00,2026-05-01 02:40:00,2026-05-01,2026-05-01
2026-04-30 18:34:00,2026-05-01 08:34:00,2026-05-01,2026-05-01
2026-04-30 10:44:00,2026-05-01 00:44:00,2026-05-01,2026-05-01
2026-04-30 09:31:00,2026-04-30 23:31:00,2026-04-30,2026-04-30
2026-04-30 12:51:00,2026-05-01 02:51:00,2026-05-01,2026-05-01
…,…,…,…
2026-05-01 15:44:32,2026-05-02 05:44:32,2026-05-02,2026-05-02
2026-05-01 07:50:29,2026-05-01 21:50:29,2026-05-01,2026-05-01
2026-05-01 13:46:26,2026-05-02 03:46:26,2026-05-02,2026-05-01


In [55]:
performance_processed = performance_updated_ns.with_columns([
    (1 - pl.col("Promoter Score (Calc)") - pl.col("Detractor Score (Calc)"))
        .alias("Neutral Score (Calc)")
])

performance_processed = performance_processed.with_columns([
    (pl.col("Promoter Score (Calc)") * pl.col("Survey Submitted (Count)")).round(0).alias("_promoter"),
    (pl.col("Detractor Score (Calc)") * pl.col("Survey Submitted (Count)")).round(0).alias("_detractor"),
    (pl.col("Neutral Score (Calc)") * pl.col("Survey Submitted (Count)")).round(0).alias("_neutral")
])

performance_processed = performance_processed.with_columns([
    pl.when(pl.col("Initiated Outbound (Yes / No)") == "Yes").then(1).otherwise(0).alias("_aob"),

    pl.col("Survey Submitted (Count)").alias("_survey"),
    pl.col("CCR72").cast(pl.Float64).fill_null(0.0).alias("_fup_72"),
    (1.0 - pl.col("CCR72").cast(pl.Float64).fill_null(0.0)).alias("_rr"),

    pl.col("Joined Time").dt.date().alias("_PST.Date"),
    pl.col("Joined Time").dt.strftime("%y_%m").alias("_PST.Month"),
    pl.col("Joined Time").dt.week().alias("_PST.Week"),
    pl.col("Joined Time").dt.year().alias("_PST.Year"),

    pl.concat_str([
        pl.col("Agent Email ID").cast(pl.Utf8).fill_null(""),
        pl.col("Conversation Id").cast(pl.Utf8).fill_null(""),
        pl.col("Joined Time").dt.strftime("%y%m%d%H%M%S")
    ], separator="_").alias("_conver_unique"),

    pl.when(pl.col("Group").is_in(["Non_Lodging", "Lodging"])).then(pl.lit("agent")).otherwise(None).alias("Agent"),
    pl.col("_Date_Converted").alias("_Date"),
    pl.col("Survey Offered (Count)").alias("_offer"),
    pl.when(pl.col("Group") == "Lodging").then(pl.lit("LG Tenure"))
     .when(pl.col("Group") == "Non_Lodging").then(pl.lit("NL Tenure"))
     .otherwise(None)
     .alias("Tenure")
])
performance_processed = performance_processed.with_columns(
    (
        pl.col("_PST.Date").cast(pl.Date) -
        pl.col("Production Start date")
          .cast(pl.String)
          .str.to_date("%Y-%m-%d %H:%M:%S", strict=False)
          .cast(pl.Date)
    )
    .dt.total_days()
    .cast(pl.Int32)
    .alias("AON_Days")
)
performance_processed = performance_processed.with_columns([
    pl.when(
        pl.col("AON_Days").is_null() & (
            (pl.col("Agent/Non Agent") == "Agent") | (pl.col("Agent/Non Agent") == "ID Deleted")
        )
    )
    .then(pl.lit("Nesting"))
    .when(pl.col("AON_Days") > 180)
      .then(pl.lit("> 180 Days"))
    .when(pl.col("AON_Days") >= 91)
      .then(pl.lit("91 - 180"))
    .when(pl.col("AON_Days") >= 61)
      .then(pl.lit("61 - 90"))
    .when(pl.col("AON_Days") >= 31)
      .then(pl.lit("31 - 60"))
    .when(pl.col("AON_Days") >= 0)
      .then(pl.lit("00 - 30"))
    .otherwise(None)
    .alias("AON Status")
])
performance_processed = performance_processed.join_asof(
    lc_mapping,
    left_on="_PST.Date",
    right_on="Effective Date",
    by="LOB",
    strategy="backward"
)
performance_processed = performance_processed.with_columns([
    (pl.col("Handle Time (Sum)") >= pl.col("Threshole_LC")).cast(pl.Int8).alias("_lc")
])
performance_processed = performance_processed.with_columns([
    (pl.col("Handle Time (Sum)") < 240).cast(pl.Int8).alias("Short Chat")
])
performance_processed = performance_processed.with_columns([
    pl.concat_str([pl.col("Agent Email ID"), pl.col("_PST.Date").dt.strftime("%y%m%d")]).alias("KEY")
])
performance_processed = performance_processed.with_columns([
    pl.concat_str([pl.col("Agent Email ID").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],separator="_").alias("EmailID_ConversationID_KEY"),
    pl.concat_str([pl.col("OracleID").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],separator="_").alias("OracleID_ConversationID_KEY"),
    pl.concat_str([pl.col("Agent People Id").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],separator="_").alias("PeopleID_ConversationID_KEY"),
])
performance_processed = performance_processed.with_columns([
    (
        pl.when(pl.col("_promoter") > 0)
          .then(pl.lit("promoter, "))
          .otherwise(pl.lit(""))
      +
        pl.when(pl.col("_detractor") > 0)
          .then(pl.lit("detractor, "))
          .otherwise(pl.lit(""))
      +
        pl.when(pl.col("_neutral") > 0)
          .then(pl.lit("neutral"))
          .otherwise(pl.lit(""))
    ).str.strip_chars(", ").alias("_nps_type")
])

def get_30min_interval(dt):
    if dt is None:
        return None
    hour = dt.hour
    minute = dt.minute
    if minute < 30:
        start = datetime(dt.year, dt.month, dt.day, hour, 0)
        end = start + timedelta(minutes=29)
    else:
        start = datetime(dt.year, dt.month, dt.day, hour, 30)
        end = start + timedelta(minutes=29)
    return f"{start.strftime('%H:%M')}-{end.strftime('%H:%M')}"

def get_period(dt):
    if dt is None:
        return None
    hour = dt.hour
    if hour >= 18:
        return 'Night'
    elif hour >= 12:
        return 'Mid'
    else:
        return 'Morning'

performance_processed = performance_processed.with_columns([
    pl.col('Joined Time').map_elements(get_30min_interval, return_dtype=pl.Utf8).alias('_PST.Interval'),
    pl.col('Join Time (VNT)').map_elements(get_30min_interval, return_dtype=pl.Utf8).alias('_VNT.Interval'),
    pl.col('Datetime_First_Start_Shift').map_elements(get_period, return_dtype=pl.Utf8).alias('_VNT.Period')
])

performance_merged_survey_t3 = (
    performance_processed
    .join(survey_ir, on = "Conversation Id", how="left")
    .join(survey_ae, on = "Conversation Id", how="left")
    .join(survey_duet, on = "Conversation Id", how="left")
    .join(t3_final, on = "Conversation Id", how="left")
    .join(delayed_closure, on = ["Conversation Id"], how="left")
    .join(RE_DIRECT_INPUT, on = "PeopleID_ConversationID_KEY", how="left")
    .join(verbatim, on = ["_conver_unique"], how="left")
    .join(afcr_input, on = ["Agent Email ID", "Conversation Id"], how="left")
)

print(performance_merged_survey_t3.columns)
print(performance_merged_survey_t3.select(["_PST.Date", "Production Start date", "AON_Days"]).head())

selected_columns = [
    "Export Time","File Name", "Agent People Id", "Business Segment Name", "Partner Name", "Customer Loyalty Status", 
    "Response Count", "Response Time", "Latest VA Product", "Language", "Latest VA Intent", "Conversation Id",
    "Agent Queue Group Name","Joined Time","_PST.Interval", "Agent Email ID","Left Time", "Handle (Count)", "Handle Time (Sum)", "Hold Time (Sum)", "Talk Time (Sum)",
    "Join Time (VNT)","_VNT.Interval","_VNT.Period","Wrap Up Time (Sum)", "Agent Business Location",
    "CCR72", "_PST.Date", "_PST.Month", "_PST.Year", "_aob", "LOB", "_conver_unique", "_nps_type", "_promoter", "_detractor", "_neutral", 
    "_survey", "_ir", "_ae", "d_happy_response", "d_surprised_response", "e_response", "t_response", "u_response", 'delight', 'usability', 'ease', 'trust', 'DUET', 
    "_offer", "T3", "Re_Direct", "Re-Direct Text", "Exceed Time", "Exceed Chat", "Agent Disconnect", "Ghost", "Requeued", "_verbatim",
    "Exceed Bucket", "_PST.Week", "_lc", "AON Status", "Agent/Non Agent", "Tenure", "OracleID", "People ID", 
    "IEX ID", "Employee Name", "Alias", "Designation", "Detail Status", "Active", "TL ID", "Supervisor Email", "Supervisor Name",
    "Wave", "Group", "_Date", "Mini TL - Email", "Mini TL", "Mini TL Start Date","Site", "Traveler Unresponsive", "Short Chat",  "Passed Sessions", "Failed Sessions", "Total Sessions"
]

fcr_columns = [
    "LOB","OracleID_ConversationID_KEY","EmailID_ConversationID_KEY","_nps_type","_detractor","_survey","_PST.Week","AON Status",
    "Agent/Non Agent","Tenure","Employee Name","Alias","Designation","Detail Status","TL ID","Supervisor Name", "Supervisor Email",
    "Wave","Group","_Date","Mini TL - Email","Mini TL","Mini TL Start Date","Site","Agent Business Location"
]

missing_cols = [col for col in selected_columns if col not in performance_merged_survey_t3.columns]
print("Các cột bị thiếu:", missing_cols)

performance_filtered = performance_merged_survey_t3.select(selected_columns)

performance_filtered = performance_filtered.unique()

performance_filtered = performance_filtered.sort(
    by=["Conversation Id", "Agent Email ID", "Joined Time"],
    descending=[False, False, True]
).with_columns(
    (pl.col("Joined Time").cum_count().over(["Conversation Id", "Agent Email ID"]) > 1)
    .cast(pl.Int8)
    .alias("Duplicate_Flag")
)

C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_26692\1670822369.py:70: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  performance_processed = performance_processed.join_asof(


['Joined Date', 'Agent People Id', 'Conversation Id', 'Agent Email ID', 'Agent Queue Group Name', 'Latest VA Intent', 'Agent Business Location', 'Initiated Outbound (Yes / No)', 'Latest VA Product', 'Response Count', 'Response Time', 'Customer Loyalty Status', 'Business Segment Name', 'Partner Name', 'Language', 'Effective Disconnected By', 'Joined Time', 'Left Time', 'Entry Point App', 'Device Category', 'Device Os', 'Browser', 'Has Followup Time', 'Queue Name', 'Agent Manager Name', 'Agent Name', 'Handle (Count)', 'Handle Time (Sum)', 'Wrap Up Time (Sum)', 'Hold Time (Sum)', 'Talk Time (Sum)', 'Follow Up (Count)', 'Survey Submitted (Count)', 'Promoter Score (Calc)', 'Detractor Score (Calc)', 'Assigned Agent Time (Sum)', 'Sum of Chat Agent First Response Time', 'Survey Offered (Count)', 'File Name', 'Export Time', 'CCR72', 'Join Time (VNT)', 'Join Date (VNT)', 'OracleID', 'People ID', 'IEX ID', 'Employee Name', 'Alias', 'Designation', 'Detail Status', 'Active', 'TL ID', 'Supervisor Em

In [56]:
# # 1) Read the removal list from Excel
# removed_id = pl.read_excel(folder_paths["mapping_file"], sheet_name="id_removed")

# # 2) Helper to normalize Conversation Id (cast -> drop quotes -> trim spaces)
# def clean_id(expr: pl.Expr) -> pl.Expr:
#     return (
#         expr.cast(pl.Utf8)
#             .str.replace_all(r'["“”]', "")      # remove any quote characters
#             .str.replace_all(r'^\s+|\s+$', "")  # trim leading/trailing whitespace (regex)
#     )

# # Validate required column
# if "Conversation Id" not in removed_id.columns:
#     raise ValueError("Sheet 'id_removed' must contain a 'Conversation Id' column.")

# # Canonical removal list: unique, non-null, non-empty
# removed_id_norm = (
#     removed_id
#     .select(clean_id(pl.col("Conversation Id")).alias("Conversation Id"))
#     .drop_nulls()
#     .filter(pl.col("Conversation Id") != "")
#     .unique()
# )

# # 3) Instead of removing rows, add a flag column "IDs Removed" (Yes/No)
# performance_filtered = (
#     performance_filtered
#     .with_columns(clean_id(pl.col("Conversation Id")).alias("__conv_norm"))
#     .join(
#         removed_id_norm
#             .select(pl.col("Conversation Id").alias("__conv_norm"))
#             .with_columns(pl.lit(1).alias("__rm")),
#         on="__conv_norm",
#         how="left",
#     )
#     .with_columns(pl.col("__rm").fill_null(0).alias("IDs Removed"))
#     .drop(["__conv_norm", "__rm"])
# )

# # (Optional) thống kê nhanh
# flagged = performance_filtered.select(pl.col("IDs Removed").sum().alias("flagged")).item()
# total = performance_filtered.height
# print(f"Flagged (IDs Removed=1): {flagged} / {total} rows")

In [57]:
# # Count affected rows and update _detractor

# affected = performance_filtered.filter(pl.col("_detractor") > 3).height
# performance_filtered = performance_filtered.with_columns(
#     pl.when(pl.col("_detractor") > 3).then(1).otherwise(pl.col("_detractor")).alias("_detractor")
# )

# print(f"Updated _detractor to 1 for {affected} row(s) where _detractor > 3.")

In [58]:
performance_filtered = performance_filtered.sort(
    by=["Conversation Id", "Agent Email ID", "Joined Time"],
    descending=[False, False, True]
).with_columns(
    (pl.col("Joined Time").cum_count().over(["Conversation Id", "Agent Email ID"]) > 1)
    .cast(pl.Int8)
    .alias("IDs Removed")
)

In [59]:
# -------------------------------------------------------------------------------------
# NEW LOGIC: Handle duplicate DUET = 0 cases
# Logic: If a Conversation Id has more than 3 rows with DUET = 0:
# -> Keep only the 1st row as 0.
# -> Set the remaining rows (2nd, 3rd, etc.) to null.
# -> If the count of zeros is <= 3, keep them as is.
# -------------------------------------------------------------------------------------

# 1. Sort to ensure deterministic order for ranking (essential for cum_sum)
performance_filtered = performance_filtered.sort(["Conversation Id"])

# 2. Create helper columns for calculation
performance_filtered = performance_filtered.with_columns([
    # Flag rows where DUET is exactly 0
    (pl.col("DUET") == 0).alias("_is_zero")
])

performance_filtered = performance_filtered.with_columns([
    # Count total number of DUET=0 rows per Conversation Id
    pl.col("_is_zero").sum().over("Conversation Id").alias("_zero_total"),
    
    # Calculate cumulative rank for DUET=0 rows (1, 2, 3...) within each Conversation Id
    pl.col("_is_zero").cum_sum().over("Conversation Id").alias("_zero_rank")
])

# 3. Apply the replacement logic
performance_filtered = performance_filtered.with_columns(
    pl.when(
        (pl.col("_is_zero")) &          # Target only rows where DUET is 0
        (pl.col("_zero_total") > 3) &   # Condition: Total zeros for this ID > 3
        (pl.col("_zero_rank") > 1)      # Condition: Skip the first one (keep rank 1), nullify rank > 1
    )
    .then(None)                         # Set to null
    .otherwise(pl.col("DUET"))          # Otherwise keep original value
    .alias("DUET")
).drop(["_is_zero", "_zero_total", "_zero_rank"]) # Drop helper columns

# (Optional) Print check
# print("Rows set to null due to DUET=0 duplicates > 3 rule:", performance_filtered.filter(pl.col("DUET").is_null()).height)
# -------------------------------------------------------------------------------------

In [60]:
performance_unique = performance_filtered.drop(["Export Time", "File Name"])
performance_unique = performance_unique.unique()

performance_hcm = performance_unique.filter(pl.col("Agent Business Location").str.contains("Ho Chi Minh", literal=True))
performance_all_site = performance_unique

In [61]:
for month, group in performance_hcm.group_by('_PST.Month'):
    month_value = month[0]
    file_name = f"{month_value}.csv"
    file_path = os.path.join(folder_paths["output_miv_performance"], file_name)
    group.write_csv(file_path)

parquet_file = "_miv_performance_hcm.parquet"
out_path_parquet = os.path.join(folder_paths["output_miv_performance"], parquet_file)
performance_hcm_parquet = performance_hcm.write_parquet(out_path_parquet)

In [62]:
print(performance_all_site['_PST.Month'].unique())

shape: (2,)
Series: '_PST.Month' [str]
[
	"26_04"
	"26_05"
]


In [63]:
print(performance_all_site.schema)

Schema({'Agent People Id': String, 'Business Segment Name': String, 'Partner Name': String, 'Customer Loyalty Status': String, 'Response Count': String, 'Response Time': String, 'Latest VA Product': String, 'Language': String, 'Latest VA Intent': String, 'Conversation Id': String, 'Agent Queue Group Name': String, 'Joined Time': Datetime(time_unit='us', time_zone=None), '_PST.Interval': String, 'Agent Email ID': String, 'Left Time': String, 'Handle (Count)': Int64, 'Handle Time (Sum)': Float64, 'Hold Time (Sum)': Float64, 'Talk Time (Sum)': Float64, 'Join Time (VNT)': Datetime(time_unit='us', time_zone=None), '_VNT.Interval': String, '_VNT.Period': String, 'Wrap Up Time (Sum)': Float64, 'Agent Business Location': String, 'CCR72': Float64, '_PST.Date': Date, '_PST.Month': String, '_PST.Year': Int32, '_aob': Int32, 'LOB': String, '_conver_unique': String, '_nps_type': String, '_promoter': Float64, '_detractor': Float64, '_neutral': Float64, '_survey': Float64, '_ir': Int32, '_ae': Int64,